In [1]:
import os
import sys

cwd = os.getcwd()
candidates = [
    cwd,
    os.path.join(cwd, "src"),
    os.path.join(cwd, ".."),
    os.path.join(cwd, "..", "src"),
]
for candidate in candidates:
    candidate = os.path.abspath(candidate)
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from utils.get_safest_route_v2 import (
    generate_diverse_routes, 
    score_routes_for_safety, 
    select_safest_then_fastest, 
    select_lexicographic
)
from utils.get_safest_route_v1 import safest_route
from utils.get_routes_converter import routes_converter

In [2]:
import joblib
import pandas as pd

In [3]:
G = joblib.load("../graph/chicago_walk_graph.joblib")

In [4]:
df = pd.read_csv("../dataset/final_data.csv")
df["Date"] = pd.to_datetime(df["Date"])

/var/folders/_m/x30w5wkd6l59g090_2t6sv1w0000gn/T/ipykernel_2269/3722579995.py:1: DtypeWarning: Columns (0: IUCR, 1: FBI Code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../dataset/final_data.csv")
/var/folders/_m/x30w5wkd6l59g090_2t6sv1w0000gn/T/ipykernel_2269/3722579995.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


In [5]:
df_sorted_by_latitude_longitude = df.sort_values(by=['Latitude', 'Longitude'])

In [6]:
lat1, lon1 = df_sorted_by_latitude_longitude['Latitude'].iloc[1], df_sorted_by_latitude_longitude['Longitude'].iloc[0] 
lat2, lon2 = df_sorted_by_latitude_longitude['Latitude'].iloc[9], df_sorted_by_latitude_longitude['Longitude'].iloc[5] 

print('lat 1, lon 1: ', lat1, lon1)
print('lat 2, lon 2: ', lat2, lon2)

lat 1, lon 1:  41.644606566 -87.610728247
lat 2, lon 2:  41.645795847 -87.615192567


In [7]:
t_query = pd.Timestamp("2026-07-27 23:00:00")  # 9pm — worth testing at a "riskier" hour

In [8]:
candidates = generate_diverse_routes(
	G, lat1=lat1, lon1=lon1, lat2=lat2, lon2=lon2,
	n_routes=50, penalty_factor=10, weight="length",
)
print(f"Generated {len(candidates)} diverse candidate routes")

Generated 12 diverse candidate routes


In [9]:
routes_converted = routes_converter(candidates, G, densify_every_m=50.0)

Route 0: 5 unique intersection points, cost=282.2m
  -> densified to 6 points (every ~50.0m along the actual walk path)
Route 1: 9 unique intersection points, cost=321.2m
  -> densified to 10 points (every ~50.0m along the actual walk path)
Route 2: 15 unique intersection points, cost=710.0m
  -> densified to 21 points (every ~50.0m along the actual walk path)
Route 3: 20 unique intersection points, cost=856.2m
  -> densified to 24 points (every ~50.0m along the actual walk path)
Route 4: 8 unique intersection points, cost=310.7m
  -> densified to 9 points (every ~50.0m along the actual walk path)
Route 5: 6 unique intersection points, cost=292.6m
  -> densified to 7 points (every ~50.0m along the actual walk path)
Route 6: 18 unique intersection points, cost=844.6m
  -> densified to 24 points (every ~50.0m along the actual walk path)
Route 7: 15 unique intersection points, cost=711.6m
  -> densified to 21 points (every ~50.0m along the actual walk path)
Route 8: 11 unique intersection

In [11]:
result = safest_route(
     routes=routes_converted, 
     crime_df=df_sorted_by_latitude_longitude, 
     t_query=t_query, 
     bw_space=300, 
     alpha=0.7, 
     beta=0.3, 
     debug=True)

[debug] Temporal_Modifier = 1.236509 (static aggregation, same for every route)
[debug] Spatial rescale bounds: lo=35.0396 hi=64.1795
[debug] route=0 point=0 lat=41.64469 lon=-87.61323 base_severity=39.00 spatial_raw=60.7276 R_i=66.6233
[debug] route=0 point=1 lat=41.64469 lon=-87.61426 base_severity=39.00 spatial_raw=60.7481 R_i=66.6571
[debug] route=0 point=2 lat=41.64469 lon=-87.61435 base_severity=39.00 spatial_raw=60.7532 R_i=66.6657
[debug] route=0 point=3 lat=41.64509 lon=-87.61489 base_severity=40.00 spatial_raw=60.7566 R_i=68.3807
[debug] route=0 point=4 lat=41.64549 lon=-87.61543 base_severity=40.00 spatial_raw=60.5729 R_i=68.0690
[debug] route=0 point=5 lat=41.64576 lon=-87.61481 base_severity=40.00 spatial_raw=60.4228 R_i=67.8141
[debug] route=1 point=0 lat=41.64469 lon=-87.61323 base_severity=39.00 spatial_raw=60.7276 R_i=66.6233
[debug] route=1 point=1 lat=41.64469 lon=-87.61426 base_severity=39.00 spatial_raw=60.7481 R_i=66.6571
[debug] route=1 point=2 lat=41.64462 lon=-

In [12]:
print("\n=== FINAL RESULT ===")
print("Safest route index:", result["safest_route_index"])
print(f"  Mean risk: {result['R_route_mean']:.4f}")
print(f"  Max risk:  {result['R_route_max']:.4f}")
print(f"  Combined:  {result['combined_score']:.4f}")
print("\nAll paths:")
for s in result["all_scores"]:
    print(f"  Route {s['route_index']}:  mean={s['R_route_mean']:.4f}  "
          f"max={s['R_route_max']:.4f}  combined={s['combined_score']:.4f}"
        #   //f"  point_risks={[f'{x:.4f}' for x in s['point_risks']]}"
          )


=== FINAL RESULT ===
Safest route index: 9
  Mean risk: 67.4845
  Max risk:  68.3807
  Combined:  67.7533

All paths:
  Route 0:  mean=67.6178  max=68.3807  combined=67.8467
  Route 1:  mean=67.5601  max=68.3804  combined=67.8062
  Route 2:  mean=97.5100  max=141.1873  combined=110.6132
  Route 3:  mean=98.2693  max=143.0910  combined=111.7158
  Route 4:  mean=67.5912  max=68.3804  combined=67.8279
  Route 5:  mean=67.5828  max=68.3807  combined=67.8221
  Route 6:  mean=101.8729  max=143.0910  combined=114.2384
  Route 7:  mean=96.6995  max=141.1873  combined=110.0458
  Route 8:  mean=98.0724  max=140.7319  combined=110.8703
  Route 9:  mean=67.4845  max=68.3807  combined=67.7533
  Route 10:  mean=97.8119  max=143.0910  combined=111.3957
  Route 11:  mean=99.9418  max=141.1873  combined=112.3155


In [13]:
from utils.visualize_route import visualize_safest_result

visualize_safest_result(result, routes_converted, save_path="safest_route_v2.html")

Saved to safest_route_v2.html
